# Workout Time Series (TF forecast)

# Workout Time Series — TF changepoint / 1RM forecast

Forecast per-exercise 1RM / volume over time from de-identified
`workouts.csv` (Django `export_ai_training_data`). Consumed by
`/api/v1/workout/analyze` — same JSON contract (exercise, weight, reps, sets).

In [ ]:
%pip install -q tensorflow
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent / 'training'))
from tf_utils import set_memory_growth
set_memory_growth()

In [ ]:
import pandas as pd
w = pd.read_csv('../data/user/workouts.csv')
w['date'] = pd.to_datetime(w['date'])
print(w.groupby('exercise')['weight_kg'].count().sort_values(ascending=False).head(10))

In [ ]:
# Group by (user, exercise) -> daily max 1RM proxy
g = w.dropna(subset=['weight_kg','reps']).groupby(['user_id','exercise','date'])['weight_kg'].max().reset_index()
g.head(5)

In [ ]:
import tensorflow as tf
# Simple LSTM forecast per (user, exercise) sliding window
inp = tf.keras.Input(shape=(8, 1))
x = tf.keras.layers.LSTM(32)(inp)
x = tf.keras.layers.Dense(16, activation='relu')(x)
out = tf.keras.layers.Dense(1)(x)
m = tf.keras.Model(inp, out)
m.compile('adam', 'mse')
m.summary()

In [ ]:
# build windows of 8 previous sessions -> next 1RM, per series; then:
# m.fit(X_windows, y_next, epochs=30)
print('train LSTM on 1RM windows')

In [ ]:
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log
onnx = export_keras_onnx(m, Path('../models'), 'workout_forecast', '1.0.0')
mlflow_log({'name':'workout_forecast','version':'1.0.0','artifact_path':str(onnx),
            'framework':'tensorflow','metrics':{}})